In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse

# ── Load spatial data ─────────────────────────────────────────────────
adata = sc.read_h5ad(
    '/path/to/data/'
    'Analyses_Erickson_Junyi/Thesis_Raw_Combined_Master.h5ad'
)
adata_mean = sc.read_h5ad(
    '/path/to/data/'
    'Cell2location/spatial_mapping/exploration_mean/'
    'spatial_compositional_space_mean.h5ad'
)

# ── Subset adata to QC-passed spots only ──────────────────────────────
adata = adata[adata_mean.obs_names].copy()
print(f"adata after subsetting: {adata.n_obs} spots")  # should be 101490

# ── Transfer niche labels safely ──────────────────────────────────────
adata.obs['niche'] = adata_mean.obs.loc[adata.obs_names, 'joint_alpha_0.9']
print('Niche distribution:')
print(adata.obs['niche'].value_counts().sort_index())

# ── Normalise ─────────────────────────────────────────────────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ── Load LMC gene sets ────────────────────────────────────────────────
lmc_genes = pd.read_csv(
    '/path/to/data/'
    'LMCs_genes_prostate.tsv',
    sep='\t'
)

# ── Check gene coverage ───────────────────────────────────────────────
spatial_genes = set(adata.var_names)
print('\nLMC gene coverage in spatial data:')
for lmc in sorted(lmc_genes['lmc'].unique()):
    lmc_gene_set = set(lmc_genes[lmc_genes['lmc'] == lmc]['gene'])
    overlap = lmc_gene_set & spatial_genes
    pct = 100 * len(overlap) / len(lmc_gene_set)
    print(f'  {lmc}: {len(overlap)}/{len(lmc_gene_set)} ({pct:.1f}%)')

# ── Manual vectorised AUCell ──────────────────────────────────────────
def aucell_score_fast(adata, gene_sets, n_genes_ranked=None, batch_size=5000):
    X = adata.X
    if issparse(X):
        X = X.toarray().astype(np.float32)  # float32 halves memory vs float64

    gene_names = np.array(adata.var_names)
    n_spots, n_genes = X.shape

    if n_genes_ranked is None:
        n_genes_ranked = int(np.ceil(0.05 * n_genes))

    print(f'Spots: {n_spots}, Genes: {n_genes}, Threshold: top {n_genes_ranked}')

    # Pre-compute gene indices per set once
    set_gene_idx = {}
    for set_name, genes in gene_sets.items():
        valid = [g for g in genes if g in gene_names]
        if len(valid) == 0:
            print(f'WARNING: {set_name} — no genes found')
            continue
        idx = np.where(np.isin(gene_names, valid))[0]
        set_gene_idx[set_name] = (idx, len(genes))
        print(f'{set_name}: {len(valid)}/{len(genes)} genes found')

    # Initialise score arrays
    scores = {name: np.zeros(n_spots, dtype=np.float32)
              for name in set_gene_idx}

    # Process in batches — never build full top_mask
    n_batches = int(np.ceil(n_spots / batch_size))
    for b in range(n_batches):
        start = b * batch_size
        end = min(start + batch_size, n_spots)
        if b % 5 == 0:
            print(f'Batch {b+1}/{n_batches} (spots {start}-{end})')

        X_batch = X[start:end]  # shape: (batch_size, n_genes)

        # Rank genes for this batch only
        ranks = np.argsort(-X_batch, axis=1)  # shape: (batch_size, n_genes)

        # Binary top mask for this batch only
        batch_n = end - start
        top_mask = np.zeros((batch_n, n_genes), dtype=bool)
        top_mask[np.arange(batch_n)[:, None], ranks[:, :n_genes_ranked]] = True

        for set_name, (gene_idx, n_total) in set_gene_idx.items():
            scores[set_name][start:end] = (
                top_mask[:, gene_idx].sum(axis=1) / n_total
            )

        del X_batch, ranks, top_mask  # free batch memory immediately

    return pd.DataFrame(scores, index=adata.obs_names)

# ── Build gene set dict ───────────────────────────────────────────────
gene_sets = {
    lmc: lmc_genes[lmc_genes['lmc'] == lmc]['gene'].tolist()
    for lmc in sorted(lmc_genes['lmc'].unique())
}

# ── Run scoring ───────────────────────────────────────────────────────
print('\nRunning AUCell scoring...')
lmc_scores_df = aucell_score_fast(adata, gene_sets)
print(lmc_scores_df.describe().round(4))

# ── Add metadata and save per-spot scores ─────────────────────────────
lmc_cols = sorted(gene_sets.keys())
lmc_scores_df['niche'] = adata.obs['niche'].values
lmc_scores_df['study'] = adata.obs['study'].values
lmc_scores_df.to_csv('LMC_scores_per_spot.csv')
print('Saved per-spot scores')

# ── Aggregate by niche ────────────────────────────────────────────────
niche_mean_scores = lmc_scores_df.groupby('niche')[lmc_cols].mean()
print('\nMean AUCell scores per niche:')
print(niche_mean_scores.round(4).to_string())
niche_mean_scores.to_csv('LMC_scores_per_niche.csv')

# ── Heatmap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    niche_mean_scores,
    cmap='YlOrRd',
    annot=True, fmt='.3f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Mean AUCell score'}
)
ax.set_title(
    'LMC programme activity per spatial niche\n'
    '(AUCell scores — higher = more active)',
    fontsize=11
)
ax.set_xlabel('LMC')
ax.set_ylabel('Spatial niche')
plt.tight_layout()
plt.savefig('LMC_activity_per_spatial_niche.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved heatmap')

good coverage across LMCs

In [ ]:
from scipy.stats import zscore
import matplotlib.pyplot as plt
import seaborn as sns

# Z-score each LMC column across niches
niche_scores_z = niche_mean_scores.apply(zscore, axis=0)

# ── Raw scores with dendrogram ────────────────────────────────────────
g1 = sns.clustermap(
    niche_mean_scores,
    cmap='YlOrRd',
    annot=True, fmt='.3f',
    linewidths=0.5,
    figsize=(16, 7),
    dendrogram_ratio=0.15,
    cbar_pos=(1.02, 0.3, 0.03, 0.4),
    cbar_kws={'label': 'Mean AUCell score'}
)
g1.ax_heatmap.set_title('LMC activity per spatial niche (raw)', fontsize=11)
g1.ax_heatmap.set_xlabel('LMC')
g1.ax_heatmap.set_ylabel('Spatial niche')
plt.savefig('LMC_activity_raw_clustermap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved raw clustermap')

# ── Z-scored with dendrogram ──────────────────────────────────────────
g2 = sns.clustermap(
    niche_scores_z,
    cmap='RdBu_r',
    center=0,
    annot=True, fmt='.2f',
    linewidths=0.5,
    figsize=(16, 7),
    dendrogram_ratio=0.15,
    cbar_pos=(1.02, 0.3, 0.03, 0.4),
    cbar_kws={'label': 'Z-score across niches'}
)
g2.ax_heatmap.set_title('LMC activity per spatial niche (z-scored)', fontsize=11)
g2.ax_heatmap.set_xlabel('LMC')
g2.ax_heatmap.set_ylabel('Spatial niche')
plt.savefig('LMC_activity_zscore_clustermap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved z-scored clustermap')

redo everything but with niche names:

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse

# ── Load spatial data ─────────────────────────────────────────────────
adata = sc.read_h5ad(
    '/path/to/data/'
    'Analyses_Erickson_Junyi/Thesis_Raw_Combined_Master.h5ad'
)
adata_mean = sc.read_h5ad(
    '/path/to/data/'
    'Cell2location/spatial_mapping/exploration_mean/'
    'spatial_compositional_space_mean.h5ad'
)

# ── Subset adata to QC-passed spots only ──────────────────────────────
adata = adata[adata_mean.obs_names].copy()
print(f"adata after subsetting: {adata.n_obs} spots")  # should be 101490

# ── Transfer niche labels safely ──────────────────────────────────────
adata.obs['niche'] = adata_mean.obs.loc[adata.obs_names, 'joint_alpha_0.9']
print('Niche distribution:')
print(adata.obs['niche'].value_counts().sort_index())

# ── Normalise ─────────────────────────────────────────────────────────
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# ── Load LMC gene sets ────────────────────────────────────────────────
lmc_genes = pd.read_csv(
    '/path/to/data/'
    'LMCs_genes_prostate.tsv',
    sep='\t'
)

# ── Check gene coverage ───────────────────────────────────────────────
spatial_genes = set(adata.var_names)
print('\nLMC gene coverage in spatial data:')
for lmc in sorted(lmc_genes['lmc'].unique()):
    lmc_gene_set = set(lmc_genes[lmc_genes['lmc'] == lmc]['gene'])
    overlap = lmc_gene_set & spatial_genes
    pct = 100 * len(overlap) / len(lmc_gene_set)
    print(f'  {lmc}: {len(overlap)}/{len(lmc_gene_set)} ({pct:.1f}%)')

# ── Manual vectorised AUCell ──────────────────────────────────────────
def aucell_score_fast(adata, gene_sets, n_genes_ranked=None, batch_size=5000):
    X = adata.X
    if issparse(X):
        X = X.toarray().astype(np.float32)  # float32 halves memory vs float64

    gene_names = np.array(adata.var_names)
    n_spots, n_genes = X.shape

    if n_genes_ranked is None:
        n_genes_ranked = int(np.ceil(0.05 * n_genes))

    print(f'Spots: {n_spots}, Genes: {n_genes}, Threshold: top {n_genes_ranked}')

    # Pre-compute gene indices per set once
    set_gene_idx = {}
    for set_name, genes in gene_sets.items():
        valid = [g for g in genes if g in gene_names]
        if len(valid) == 0:
            print(f'WARNING: {set_name} — no genes found')
            continue
        idx = np.where(np.isin(gene_names, valid))[0]
        set_gene_idx[set_name] = (idx, len(genes))
        print(f'{set_name}: {len(valid)}/{len(genes)} genes found')

    # Initialise score arrays
    scores = {name: np.zeros(n_spots, dtype=np.float32)
              for name in set_gene_idx}

    # Process in batches — never build full top_mask
    n_batches = int(np.ceil(n_spots / batch_size))
    for b in range(n_batches):
        start = b * batch_size
        end = min(start + batch_size, n_spots)
        if b % 5 == 0:
            print(f'Batch {b+1}/{n_batches} (spots {start}-{end})')

        X_batch = X[start:end]  # shape: (batch_size, n_genes)

        # Rank genes for this batch only
        ranks = np.argsort(-X_batch, axis=1)  # shape: (batch_size, n_genes)

        # Binary top mask for this batch only
        batch_n = end - start
        top_mask = np.zeros((batch_n, n_genes), dtype=bool)
        top_mask[np.arange(batch_n)[:, None], ranks[:, :n_genes_ranked]] = True

        for set_name, (gene_idx, n_total) in set_gene_idx.items():
            scores[set_name][start:end] = (
                top_mask[:, gene_idx].sum(axis=1) / n_total
            )

        del X_batch, ranks, top_mask  # free batch memory immediately

    return pd.DataFrame(scores, index=adata.obs_names)

# ── Build gene set dict ───────────────────────────────────────────────
gene_sets = {
    lmc: lmc_genes[lmc_genes['lmc'] == lmc]['gene'].tolist()
    for lmc in sorted(lmc_genes['lmc'].unique())
}

# ── Run scoring ───────────────────────────────────────────────────────
print('\nRunning AUCell scoring...')
lmc_scores_df = aucell_score_fast(adata, gene_sets)
print(lmc_scores_df.describe().round(4))

# ── Add metadata and save per-spot scores ─────────────────────────────
lmc_cols = sorted(gene_sets.keys())
lmc_scores_df['niche'] = adata.obs['niche'].values
lmc_scores_df['study'] = adata.obs['study'].values
lmc_scores_df.to_csv('LMC_scores_per_spot.csv')
print('Saved per-spot scores')

# ── Define descriptive names for each niche ───────────────────────────
niche_mapping = {
    0: 'Niche 0\nLow-quality/acellular',
    1: 'Niche 1\nPeriurethral ductal',
    2: 'Niche 2\nAggressive tumour',
    3: 'Niche 3\nNecrotic/Low-quality',
    4: 'Niche 4\nFibromuscular stroma',
    5: 'Niche 5\nIncidental cancer',
    6: 'Niche 6\nLuminal epithelium',
    '0': 'Niche 0\nLow-quality/acellular',
    '1': 'Niche 1\nPeriurethral ductal',
    '2': 'Niche 2\nAggressive tumour',
    '3': 'Niche 3\nNecrotic/Low-quality',
    '4': 'Niche 4\nFibromuscular stroma',
    '5': 'Niche 5\nIncidental cancer',
    '6': 'Niche 6\nLuminal epithelium'
}

# ── Aggregate by niche and rename index labels ────────────────────────
niche_mean_scores = lmc_scores_df.groupby('niche', observed=False)[lmc_cols].mean()
niche_mean_scores.index = niche_mean_scores.index.map(niche_mapping)

print('\nMean AUCell scores per niche:')
print(niche_mean_scores.round(4).to_string())
niche_mean_scores.to_csv('LMC_scores_per_niche.csv')

# ── Heatmap ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))  # Increased height slightly for multiline text
sns.heatmap(
    niche_mean_scores,
    cmap='YlOrRd',
    annot=True, fmt='.3f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Mean AUCell score'}
)
ax.set_title(
    'LMC programme activity per spatial niche\n'
    '(AUCell scores — higher = more active)',
    fontsize=11
)
ax.set_xlabel('LMC')
ax.set_ylabel('Spatial niche')
plt.tight_layout()
plt.savefig('LMC_activity_per_spatial_niche.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved heatmap')

In [ ]:
from scipy.stats import zscore
import matplotlib.pyplot as plt
import seaborn as sns

# Z-score each LMC column across niches
niche_scores_z = niche_mean_scores.apply(zscore, axis=0)

# ── Raw scores with dendrogram ────────────────────────────────────────
g1 = sns.clustermap(
    niche_mean_scores,
    cmap='YlOrRd',
    annot=True, fmt='.3f',
    linewidths=0.5,
    figsize=(16, 8.5),  # Expanded height slightly to give room to the y-labels
    dendrogram_ratio=0.15,
    cbar_pos=(1.02, 0.3, 0.03, 0.4),
    cbar_kws={'label': 'Mean AUCell score'}
)
g1.ax_heatmap.set_title('LMC activity per spatial niche (raw)', fontsize=11)
g1.ax_heatmap.set_xlabel('LMC')
g1.ax_heatmap.set_ylabel('Spatial niche')
plt.savefig('LMC_activity_raw_clustermap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved raw clustermap')

# ── Z-scored with dendrogram ──────────────────────────────────────────
g2 = sns.clustermap(
    niche_scores_z,
    cmap='RdBu_r',
    center=0,
    annot=True, fmt='.2f',
    linewidths=0.5,
    figsize=(16, 8.5),  # Expanded height slightly to give room to the y-labels
    dendrogram_ratio=0.15,
    cbar_pos=(1.02, 0.3, 0.03, 0.4),
    cbar_kws={'label': 'Z-score across niches'}
)
g2.ax_heatmap.set_title('LMC activity per spatial niche (z-scored)', fontsize=11)
g2.ax_heatmap.set_xlabel('LMC')
g2.ax_heatmap.set_ylabel('Spatial niche')
plt.savefig('LMC_activity_zscore_clustermap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved z-scored clustermap')